# proof

> Have other models read a document's pages again, and review the word changes they propose before any of them is applied

In [ ]:
#| default_exp proof

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import tempfile

A vision model's reading of a page is good but not perfect. One way to catch its slips is to have other models, the readers, read the same pages again and compare. Where a reading differs from the stored text, the difference is filed as a change: a run of words to replace, insert or delete, with the names of the readers who made it. Nothing in the document changes until its owner decides.

`Vault.proofread(doc_id, readers)` has each reader read the pages and files their changes. A change that at least `agree` readers (2 by default) make identically is `proposed`, and a change fewer readers make is a `doubt`. A reader named twice counts once. The owner then works through the review queue:

| method | what it does |
|---|---|
| `changes(doc_id, state)` | lists the changes, all of them or those in one state: `proposed`, `doubt`, `accepted` or `rejected` |
| `accept(doc_id, ids)` | applies changes, all the `proposed` ones when `ids` is None, as a new version of each page they touch (route `proof`), and re-indexes the document |
| `reject(doc_id, ids)` | sets changes aside; they stay in the record as not taken |
| `edit_change(doc_id, id, after)` | rewrites the proposed text of a change that is still open |
| `undo_decision(doc_id, ids)` | takes a decision back: a rejected change returns to the state it had, and an accepted change's page returns to its text before the accept, with every change that accept applied `proposed` again |
| `revert_page(doc_id, page, version)` | files any old version of a page again as its newest version (route `revert`) |

A change filed against an older version of its page cannot be accepted; `accept` raises an error that asks for a new proofread. An accept can be undone only while its page has not changed since. `edit_change`, `undo_decision` and `revert_page` are for a person only, and raise `PermissionError` with `by='agent'`. `proof_md` writes the record of a proofread as markdown, page by page: what changed, what is proposed, what is in doubt and what was not taken.

In [ ]:
#| export
import json, re, time, uuid
from itertools import accumulate
from difflib import SequenceMatcher
from fastcore.all import AttrDict, L, Path, first, ifnone, patch
from litesearch import write_txn
from vishalakshi.core import Vault
from vishalakshi.pii import _person
from vishalakshi.read import _md, read_page, render, as_list, flags, IMG_EXTS

## Word changes

`word_changes(a, b)` compares two texts word by word and returns each run of words that `b` replaces, inserts or deletes, as `{at, n, before, after}`. `at` is the index of the first word in `a`, `n` is how many of `a`'s words the change replaces (0 for an insertion), and `before` and `after` are the words themselves. `apply_changes` makes a list of such changes, from the last to the first, and keeps the text's own spacing and line breaks everywhere else.

A stored page also has blocks, and they have to stay in step with its text. `block_changes` makes each change in the block its words fall in, and splits a change that crosses the end of a block. A table's cells and a heading's `##` stay as read. `change_box` finds the `box` of the block a change falls in, so a review screen can show the change on the page image.

In [ ]:
#| export
_TOK = re.compile(r'\S+|\s+')
def _words(t): return [w for w in _TOK.findall(t or '') if not w.isspace()]

def word_changes(a:str, b:str) -> list:
    "Compare `a` and `b` word by word and return each run of words `b` changes, as `{at, n, before, after}`."
    wa, wb = _words(a), _words(b)
    return [dict(at=i1, n=i2 - i1, before=' '.join(wa[i1:i2]), after=' '.join(wb[j1:j2]))
            for op, i1, i2, j1, j2 in SequenceMatcher(None, wa, wb, autojunk=False).get_opcodes() if op != 'equal']

def change_box(blocks, text:str, at:int):
    "The `box` of the block that word `at` of `text` falls in, the last block for a position past the end, or None when the blocks do not read as `text`."
    ns = list(accumulate(len(_words(_md([b]))) for b in blocks or []))
    if not ns or ns[-1] != len(_words(text)): return None
    return first(b for b, e in zip(blocks, ns) if at < e or e == ns[-1]).get('box')

def apply_changes(t:str, changes) -> str:
    "Make `changes` in `t`, each replacing `n` words from word `at` with `after`, and keep the text's own spacing elsewhere."
    toks = _TOK.findall(t or '')
    lead, out = (toks[0] if toks and toks[0].isspace() else ''), []
    for x in toks:
        if not x.isspace(): out.append([x, ''])
        elif out: out[-1][1] = x
    for c in sorted(changes, key=lambda c: -c['at']):
        a, e = c['at'], c['at'] + c['n']
        new = [[w, ' '] for w in (c['after'] or '').split()]
        if new and c['n']: new[-1][1] = out[e - 1][1]
        elif new and a >= len(out):
            new[-1][1] = out[-1][1] if out else ''
            if out: out[-1][1] = out[-1][1] or ' '
        elif not new and c['n'] and e == len(out) and a: out[a - 1][1] = out[e - 1][1]
        out[a:e] = new
    return lead + ''.join(w + s for w, s in out)

def block_changes(blocks, text:str, changes) -> list|None:
    "Make `changes` to `text` in the blocks their words fall in, splitting a change at a block's end, or return None when the blocks do not read as `text`."
    ns = [len(_words(_md([b]))) for b in blocks or []]
    if not ns or sum(ns) != len(_words(text)): return None
    st, out = [0, *accumulate(ns)][:-1], [dict(b) for b in blocks]
    per = [[] for _ in out]
    for c in changes:
        a, e = c['at'], c['at'] + c['n']
        hit = [k for k, (s, n) in enumerate(zip(st, ns)) if max(a, s) < min(e, s + n)] if c['n'] else \
              [next((k for k, (s, n) in enumerate(zip(st, ns)) if a < s + n), len(out) - 1)]
        for j, k in enumerate(hit):
            hd = int(out[k].get('kind') == 'heading' and bool(ns[k]))
            lo, hi = max(a, st[k]) - st[k] - hd, min(e, st[k] + ns[k]) - st[k] - hd
            per[k].append(dict(at=max(lo, 0), n=max(hi, 0) - max(lo, 0), after=c['after'] if not j else ''))
    for b, cs in zip(out, per):
        if cs and not b.get('rows'): b['text'] = apply_changes(b.get('text') or '', cs)
    return out

In [ ]:
_a, _b = 'धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः', 'धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवाः ॥१॥'
_c = word_changes(_a, _b)
test_eq(_c, [dict(at=3, n=1, before='युयुत्सवः', after='युयुत्सवाः ॥१॥')])
test_eq(apply_changes(_a, _c), _b)
test_eq(apply_changes('a b c', word_changes('a b c', 'a c')), 'a c')
test_eq(apply_changes('a c', word_changes('a c', 'a b c')), 'a b c')
test_eq(apply_changes('line one\nline two', word_changes('line one\nline two', 'line 1\nline two')), 'line 1\nline two')
_bl = [dict(kind='heading', text='Title here', box=[0, 0, 1, .1]), dict(kind='prose', text='one two three', box=[0, .2, 1, .4]), dict(kind='prose', text='four')]
_tx = _md(_bl)
test_eq([change_box(_bl, _tx, i) for i in (0, 2, 3, 5, 6, 9)], [[0, 0, 1, .1], [0, 0, 1, .1], [0, .2, 1, .4], [0, .2, 1, .4], None, None])   # the heading is 3 words; past the end is the last block
test_eq((change_box(_bl, 'other words', 0), change_box(None, _tx, 0)), (None, None))
for x, y in [('', 'new words'), ('a', ''), ('x y z', 'z y x'), ('one two', 'one two three')]: test_eq(apply_changes(x, word_changes(x, y)), y if x or y else '')
test_eq([b['text'] for b in block_changes(_bl, _tx, word_changes(_tx, '## Title there\n\none 2 three\n\nfour'))], ['Title there', 'one 2 three', 'four'])
test_eq([b['text'] for b in block_changes(_bl, _tx, word_changes(_tx, '## Title here\n\none two 3 4'))], ['Title here', 'one two 3 4', ''])   # a run across a block's end splits there
test_eq([b['text'] for b in block_changes(_bl, _tx, word_changes(_tx, '## Title here\n\none two three\n\nfour five'))], ['Title here', 'one two three', 'four five'])
test_eq(block_changes(_bl, _tx, word_changes(_tx, 'Title here one two three four'))[0]['text'], 'Title here')   # the ## goes from the text, the heading stays
_tb = [dict(kind='table', rows=[['a', 'b']]), dict(kind='prose', text='after it')]
test_eq(block_changes(_tb, _md(_tb), word_changes(_md(_tb), _md(_tb).replace('a', 'x', 1).replace('after', 'later'))), [_tb[0], dict(kind='prose', text='later it')])
test_eq(block_changes(_bl, 'not the blocks', []), None)

## Proofreading

`readers` are model ids. `chats` maps a reader to a stand-in callable, which is how the tests run without models. Each reader reads the page image through `read_page`, so privacy works as it does there: a reader that may not see a page skips it, and the skip is reported in `blocked`. A hosted reader may not see a Personal page without `allow=True`, and no hosted reader sees a Local only page. Each hosted reading is written to the send log.

Running `proofread` again replaces the open changes, `proposed` and `doubt`, on each page it reads, and leaves decided ones alone, so a proofread can be repeated. A change keeps the `box` of the block it falls in when the page's blocks have boxes. `progress(done, total, page, route, cost)` is called once with `done=0` and after each page. Returning False stops the rest, and the result carries `cancelled`, `unread` and the summed `cost`.

In [ ]:
#| export
@patch
def _proof_t(self:Vault):
    t = self.db.t.proof_changes
    if not t.exists():
        with write_txn(self.db):
            t.create(id=str, doc_id=str, store=str, page=int, version=int, at=int, n=int, before=str, after=str, by=str, state=str, ts=float, box=str, was=str, pk='id')
    for c in ('box', 'was'):
        if c not in t.columns_dict: self.db.execute(f'alter table proof_changes add column {c} text')
    return t

@patch
def proofread(self:Vault, doc_id:str, readers, pages=None, agree:int=2, preset:str=None, script:str=None, hint:str=None,
              dpi:int=200, chats:dict=None,
              pii:str='redact',    # what `read_page` applies: a reader that may not see a page is skipped for it and reported in `blocked`
              allow:bool=False,    # allow once, for Personal pages
              progress=None,       # `progress(done, total, page, route, cost)` after each page; False stops the rest
) -> dict:
    "Have each of `readers` read the pages again and file their word changes to the current text, `proposed` where at least `agree` readers make the same one."
    d, readers, pages = first(self.t.docs(where='id=?', where_args=[doc_id])), list(dict.fromkeys(as_list(readers))), as_list(pages, int)   # a reader named twice is one reader
    assert d, f'no document {doc_id}'
    m, src, t = json.loads(d['meta'] or '{}'), Path(d['source']), self._proof_t()
    preset, script, hint = ifnone(preset, m.get('preset', 'generic')), ifnone(script, m.get('script', '')), ifnone(hint, m.get('hint', ''))
    cur = [p for p in self.pages(doc_id) if pages is None or p.page in set(pages)]
    n, blocked, cost, unread, stop = 0, [], 0, [], False
    def tell(*a):
        try: return progress(*a) is not False
        except Exception: return True
    if progress: stop = not tell(0, len(cur), None, None, None)
    for i, p in enumerate(cur):
        if stop: unread.append(p.page); continue
        img = src.read_bytes() if src.suffix.lower() in IMG_EXTS else render(src, p.page, dpi)
        got, lv, read, pc = {}, self.level(doc_id, page=p.page).level, 0, None
        for r in readers:
            ch = (chats or {}).get(r)
            pg = read_page(img, r, preset, d['title'], script, hint, chat=ch, pii=pii, level=lv, text=p.text, allow=allow)
            if pg.get('blocked'):
                blocked.append(dict(page=p.page, reader=r, **pg.blocked)); continue
            read += 1
            if pg.cost is not None: pc = (pc or 0) + pg.cost
            self._read_sent(doc_id, [AttrDict(page=p.page, route='model', model=getattr(ch, 'model', r))], r, ch, pii)
            for c in word_changes(p.text, pg.text): got.setdefault((c['at'], c['n'], c['after']), dict(c, by=[]))['by'].append(r)
        cost += pc or 0
        if progress: stop = not tell(i + 1, len(cur), p.page, 'model' if read else 'blocked', pc)
        if not read: continue
        with write_txn(self.db):
            self.db.execute('delete from proof_changes where doc_id=? and store=? and page=? and state in (\'proposed\', \'doubt\')', [doc_id, self.name, p.page])
            for c in got.values():
                t.insert(dict(id=uuid.uuid4().hex[:12], doc_id=doc_id, store=self.name, page=p.page, version=p.version, at=c['at'], n=c['n'],
                              before=c['before'], after=c['after'], by=json.dumps(c['by']), state='proposed' if len(c['by']) >= agree else 'doubt', ts=time.time(),
                              box=json.dumps(b) if (b := change_box(p.blocks, p.text, c['at'])) else None))
                n += 1
    return dict(doc_id=doc_id, pages=[p.page for p in cur], changes=n, proposed=len(self.changes(doc_id, 'proposed')), blocked=blocked,
                cost=cost, cancelled=bool(unread), unread=unread)

@patch
def changes(self:Vault, doc_id:str, state:str=None, page:int=None) -> L:
    "List a document's filed changes in page and word order, or only those in one `state`: `proposed`, `doubt`, `accepted` or `rejected`."
    w, a = 'doc_id=? and store=?', [doc_id, self.name]
    if state: w, a = w + ' and state=?', a + [state]
    if page is not None: w, a = w + ' and page=?', a + [page]
    return L(AttrDict(r, by=json.loads(r['by']), box=json.loads(r.get('box') or 'null')) for r in self._proof_t().rows_where(w, a, order_by='page, at'))

@patch
def reject(self:Vault, doc_id:str, ids) -> int:
    "Set changes aside as not taken, remembering the state each had so that `undo_decision` can restore it."
    ids = as_list(ids); self._proof_t()
    with write_txn(self.db):
        for i in ids: self.db.execute("update proof_changes set was=case when state in ('proposed', 'doubt') then state else was end, state='rejected' where id=?", [i])
    return len(ids)

@patch
def accept(self:Vault, doc_id:str, ids=None) -> dict:
    "Apply changes, all the `proposed` ones when `ids` is None, as a new version of each page they touch, with its blocks changed to match, and re-index the document."
    ids = as_list(ids)
    cs = self.changes(doc_id).filter(lambda c: c.id in set(ids) if ids is not None else c.state == 'proposed')
    pg = {p.page: p for p in self.pages(doc_id)}
    stale = [c.id for c in cs if pg.get(c.page) is None or pg[c.page].version != c.version]
    assert not stale, f'changes {stale} were made against an older version of their page: proofread again'
    rows = L()
    for k, g in cs.groupby(lambda c: c.page).items():
        rows.append(AttrDict(page=k, text=apply_changes(pg[k].text, g), route='proof', flags=pg[k].flags, model=None, blocks=block_changes(pg[k].blocks, pg[k].text, g)))
    if rows:
        for r in rows: r.flags = flags(r.text)
        self._file_pages(doc_id, rows, None)
        with write_txn(self.db):
            for c in cs: self._proof_t().update(dict(id=c.id, state='accepted'))
        self._reindex(doc_id)
    return dict(doc_id=doc_id, accepted=len(cs), pages=[r.page for r in rows])

@patch
def _reindex(self:Vault, doc_id):
    d, new = first(self.t.docs(where='id=?', where_args=[doc_id])), {p.page: p for p in self.pages(doc_id)}
    self.add([(k, new[k].text) for k in sorted(new)], d['title'], source=d['source'], kind=d['kind'], meta=json.loads(d['meta'] or '{}'), force=True)

def _no(kind, error, **kw): return dict(ok=False, kind=kind, error=error, **kw)

@patch
def edit_change(self:Vault, doc_id:str, id:str, after:str, by:str='person') -> dict:
    "Rewrite the proposed text of a change that is still open; only a person may."
    _person(by, 'edit a change')
    c = first(self.changes(doc_id).filter(lambda c: c.id == id))
    if c is None: return _no('not_found', f'no change {id} on this document', ids=[id])
    if c.state not in ('proposed', 'doubt'): return _no('stale', f'change {id} is {c.state}: only an open change is edited', ids=[id])
    self._proof_t().update(dict(id=id, after=str(after or '')))
    return dict(ok=True, change=first(self.changes(doc_id).filter(lambda c: c.id == id)))

@patch
def _refile(self:Vault, doc_id, page, version):
    "File `version` of `page` again as its newest version, route `revert`; False when there is no such version."
    o = first(self.pages(doc_id, history=True).filter(lambda p: p.page == page and p.version == version))
    if o is None: return False
    self._file_pages(doc_id, [AttrDict(page=page, text=o.text, route='revert', flags=flags(o.text), model=None,
                                       blocks=json.loads(o.get('blocks') or 'null'))], None)
    return True

@patch
def revert_page(self:Vault, doc_id:str, page:int, version:int, by:str='person') -> dict:
    "File an old `version` of a page as its newest version, route `revert`, and re-index the document; only a person may."
    _person(by, 'revert a page')
    page, version = int(page), int(version)
    if not self._refile(doc_id, page, version): return _no('not_found', f'page {page + 1} has no version {version}', page=page, version=version)
    self._reindex(doc_id)
    return dict(ok=True, doc_id=doc_id, page=page, version=first(self.pages(doc_id).filter(lambda p: p.page == page)).version)

@patch
def undo_decision(self:Vault, doc_id:str, ids, by:str='person') -> dict:
    "Take decisions back: a rejected change returns to its old state, and an accepted change's page returns to its text before the accept, with that accept's changes `proposed` again; only a person may."
    _person(by, 'undo a decision')
    ids, cs = as_list(ids) or [], {c.id: c for c in self.changes(doc_id)}
    if bad := [i for i in ids if i not in cs]: return _no('not_found', 'no such change on this document', ids=bad)
    if bad := [i for i in ids if cs[i].state not in ('accepted', 'rejected')]: return _no('stale', 'these changes are not decided', ids=bad)
    cur, acc = {p.page: p.version for p in self.pages(doc_id)}, [cs[i] for i in ids if cs[i].state == 'accepted']
    if bad := [c.id for c in acc if cur.get(c.page) != c.version + 1]:
        return _no('stale', 'the page changed again after that accept', ids=bad)
    pv = {(c.page, c.version) for c in acc}
    for pg, v in sorted(pv): self._refile(doc_id, pg, v)
    now = {p.page: p.version for p in self.pages(doc_id)}
    # an accept made one version of the page: undoing it takes back every change it applied
    acc = [c for c in cs.values() if c.state == 'accepted' and (c.page, c.version) in pv]
    rej = [cs[i] for i in ids if cs[i].state == 'rejected']
    with write_txn(self.db):
        for c in acc: self._proof_t().update(dict(id=c.id, state='proposed', version=now[c.page]))
        for c in rej: self._proof_t().update(dict(id=c.id, state=c.get('was') or 'proposed'))
    if pv: self._reindex(doc_id)
    return dict(ok=True, doc_id=doc_id, undone=[c.id for c in acc + rej], pages=sorted({pg for pg, _ in pv}))

@patch
def proof_md(self:Vault, doc_id:str) -> str:
    "The record of a document's proofreading as markdown, by page: what changed, what is proposed or in doubt, and what was not taken."
    d = first(self.t.docs(where='id=?', where_args=[doc_id]))
    line = lambda c: f"- p. {c.page + 1}: {c.before or '∅'} → {c.after or '∅'} ({', '.join(c.by)})"
    out = [f"# Proofread: {d['title'] if d else doc_id}"]
    for title, st in (('Changed', 'accepted'), ('Proposed', 'proposed'), ('In doubt', 'doubt'), ('Not taken', 'rejected')):
        if (cs := self.changes(doc_id, st)): out += ['', f'## {title}', '', *map(line, cs)]
    return '\n'.join(out) + '\n'

In [ ]:
from vishalakshi.read import read_doc
from pdflite import pdf_doc
class Says:
    "A reader that reads every page as `text`."
    def __init__(self, text): self.text, self.model = text, 'fake'
    def __call__(self, msg): return dict(content=json.dumps(dict(items=[dict(kind='prose', text=self.text)])))
_d = Path(tempfile.mkdtemp())
from pdf_oxide import Pdf
_fx = Path(tempfile.mkdtemp())
for _i, _t in enumerate(['# Attention Is All You Need\n\nThe dominant sequence transduction models are based on recurrent or convolutional networks.',
                         '# Background\n\nThe goal of reducing sequential computation also forms the foundation of the Extended Neural GPU.']):
    Pdf.from_markdown(_t).save(str(_fx/f'p{_i}.pdf'))
Pdf.merge([str(_fx/'p0.pdf'), str(_fx/'p1.pdf')]).save(str(_fx/'two.pdf'))
_src = _fx/'two.pdf'
_scan = _d/'scan.pdf'; _scan.write_bytes(pdf_doc(pdf_doc(_src).extract_pages_to_bytes([0])).flatten_to_images(dpi=72))
_v = Vault(Path(tempfile.mkdtemp())/'v.db', offline=True)
_o = _v.read_doc(_scan, parse='llm', chat=Says('the dominat sequence transduction models'))
_id = _o['doc_id']
_r = _v.proofread(_id, ['a', 'b', 'c'], chats=dict(a=Says('the dominant sequence transduction models'), b=Says('the dominant sequence transduction models'),
                                                   c=Says('the dominat sequence transduktion models')))
test_eq(_r['changes'], 2); test_eq(_r['proposed'], 1)
_p = _v.changes(_id, 'proposed')[0]; test_eq((_p.before, _p.after, _p.by), ('dominat', 'dominant', ['a', 'b']))
_dt = _v.changes(_id, 'doubt')[0]; test_eq((_dt.after, _dt.by), ('transduktion', ['c']))
_v.reject(_id, [_dt.id])
test_eq(_v.accept(_id)['accepted'], 1)
test_eq(_v.pages(_id)[0].text, 'the dominant sequence transduction models'); test_eq(_v.pages(_id)[0].route, 'proof')
assert 'dominant' in _v.search('dominant sequence')[0]['snippet']
_md_ = _v.proof_md(_id)
assert '## Changed' in _md_ and 'dominat → dominant (a, b)' in _md_ and '## Not taken' in _md_
_v.proofread(_id, ['a'], chats=dict(a=Says('the dominant sequence transduction model')))
_v.reparse(_id, chat=Says('a different reading entirely'))
test_fail(lambda: _v.accept(_id, [c.id for c in _v.changes(_id, 'doubt')]), contains='older version')

In [ ]:
#| hide
# a hosted reader may not see a Personal page: it is skipped and reported, and the page's proposals stand
_pv = Vault(Path(tempfile.mkdtemp())/'pv.db', offline=True)
_po = _pv.read_doc(_scan, parse='llm', chat=Says('write to ada@example.com about the dominat models'))
_pid = _po['doc_id']
_r = _pv.proofread(_pid, ['a'], chats=dict(a=Says('write to ada@example.com about the dominant models')))
test_eq((_r['changes'], [(b['page'], b['reader'], b['kind']) for b in _r['blocked']]), (0, [(0, 'a', 'pii_blocked')]))
# allowed once, it reads; a local reader needs no allowing
_r = _pv.proofread(_pid, ['a'], chats=dict(a=Says('write to ada@example.com about the dominant models')), allow=True)
test_eq((_r['changes'], _r['blocked']), (1, []))
class LocalSays(Says): runtime = 'litert'
_r = _pv.proofread(_pid, ['b'], chats=dict(b=LocalSays('write to ada@example.com about the dominant models')))
test_eq(_r['blocked'], [])
# Local only stops even an allowed hosted reader
_pv.set_level(_pid, 'local')
test_eq(_pv.proofread(_pid, ['a'], chats=dict(a=Says('x')), allow=True)['blocked'][0]['kind'], 'local_only')


In [ ]:
#| hide
# a change keeps the box of the block it falls in; progress, cancel and cost as in `read_doc`
class Boxes(Says):
    "A reader that reads every page as two boxed blocks."
    def __init__(self, a, b, cost=None): self.a, self.b, self.cost, self.model = a, b, cost, 'fake'
    def __call__(self, msg): return dict(content=json.dumps(dict(items=[dict(kind='heading', text=self.a, box=[.1, .1, .9, .2]), dict(kind='prose', text=self.b, box=[.1, .3, .9, .6])])),
                                         usage=dict(cost=self.cost))
_bv = Vault(Path(tempfile.mkdtemp())/'b.db', offline=True)
_bid = _bv.read_doc(_src, parse='llm', chat=Boxes('Attention', 'the dominat models'))['doc_id']
_seen = []
_r = _bv.proofread(_bid, ['a', 'b'], chats=dict(a=Boxes('Attention', 'the dominant models', .5), b=Boxes('Atention', 'the dominat models', .25)),
                   progress=lambda *a: _seen.append(a))
test_eq(_seen, [(0, 2, None, None, None), (1, 2, 0, 'model', .75), (2, 2, 1, 'model', .75)])
test_eq((_r['cost'], _r['cancelled'], _r['unread']), (1.5, False, []))
test_eq(sorted((c.page, c.after, c.box) for c in _bv.changes(_bid)), [(0, 'Atention', [.1, .1, .9, .2]), (0, 'dominant', [.1, .3, .9, .6]),
                                                                       (1, 'Atention', [.1, .1, .9, .2]), (1, 'dominant', [.1, .3, .9, .6])])
_r = _bv.proofread(_bid, ['a'], chats=dict(a=Says('the dominant models')), progress=lambda d, *a: d < 1)
test_eq((_r['cost'], _r['cancelled'], _r['unread']), (0, True, [1]))
test_eq([c.box for c in _bv.changes(_bid, page=0)], [[.1, .1, .9, .2], [.1, .3, .9, .6]])
test_eq(len(_bv.changes(_bid, page=1)), 2)                              # the unread page's changes stand
_bv.accept(_bid, [c.id for c in _bv.changes(_bid, page=0)])
_bv.proofread(_bid, ['a'], pages=[0], chats=dict(a=Says('the dominant model')))
test_eq([c.box for c in _bv.changes(_bid, page=0, state='doubt')], [[.1, .3, .9, .6]])   # a proofed page keeps its blocks, so its box
test_eq(_bv.proofread(_bid, ['a'], chats=dict(a=Says('x')), progress=lambda *a: False)['unread'], [0, 1])
# a vault made before boxes gains the column
_ov = Vault(Path(tempfile.mkdtemp())/'old.db', offline=True)
_ov.db.t.proof_changes.create(id=str, doc_id=str, store=str, page=int, version=int, at=int, n=int, before=str, after=str, by=str, state=str, ts=float, pk='id')
assert 'box' in _ov._proof_t().columns_dict

In [ ]:
#| hide
# a change is edited only while open; a decision is undone: a reject to what it was, an accept by reverting its page
_uv = Vault(Path(tempfile.mkdtemp())/'u.db', offline=True)
_uid = _uv.read_doc(_scan, parse='llm', chat=Says('the dominat sequence transduktion models'))['doc_id']
_uv.proofread(_uid, ['a', 'b', 'c'], chats=dict(a=Says('the dominant sequence transduction models'), b=Says('the dominant sequence transduction models'),
                                               c=Says('the dominat sequence transduktion model')))
_cs = {c.after: c for c in _uv.changes(_uid)}
_e = _uv.edit_change(_uid, _cs['dominant'].id, 'dominent')
test_eq((_e['ok'], _e['change'].after, _e['change'].state), (True, 'dominent', 'proposed'))
test_eq(_uv.edit_change(_uid, 'nope', 'x')['kind'], 'not_found')
test_fail(lambda: _uv.edit_change(_uid, _cs['dominant'].id, 'x', by='agent'), contains='person')
_uv.reject(_uid, [_cs['model'].id])
test_eq(_uv.edit_change(_uid, _cs['model'].id, 'x')['kind'], 'stale')
test_eq(_uv.undo_decision(_uid, [_cs['model'].id])['undone'], [_cs['model'].id])
test_eq(_uv.changes(_uid, 'doubt').attrgot('id'), [_cs['model'].id])        # back to doubt, not proposed
test_eq(_uv.undo_decision(_uid, [_cs['model'].id])['kind'], 'stale')       # an open change has no decision to undo
_uv.edit_change(_uid, _cs['dominant'].id, 'dominant')
_uv.accept(_uid)
test_eq(_uv.pages(_uid)[0].text, 'the dominant sequence transduction models')
test_fail(lambda: _uv.undo_decision(_uid, [_cs['dominant'].id], by='agent'), contains='person')
_u = _uv.undo_decision(_uid, [_cs['dominant'].id])                          # one id undoes the whole accept of its page
test_eq((_u['ok'], sorted(_u['undone']), _u['pages']), (True, sorted([_cs['dominant'].id, _cs['transduction'].id]), [0]))
test_eq((_uv.pages(_uid)[0].text, _uv.pages(_uid)[0].route, _uv.pages(_uid)[0].version), ('the dominat sequence transduktion models', 'revert', 3))
assert 'dominat' in _uv.search('dominat sequence')[0]['snippet']
test_eq(_uv.accept(_uid)['accepted'], 2)                                   # the undone changes may be taken again
_uv.proofread(_uid, ['a', 'b'], chats=dict(a=Says('the dominant sequence transduction model'), b=Says('the dominant sequence transduction model')))
_uv.accept(_uid)
test_eq(_uv.undo_decision(_uid, [_cs['dominant'].id])['kind'], 'stale')    # its page has moved on since
test_eq(_uv.undo_decision(_uid, ['nope'])['kind'], 'not_found')
_rv = _uv.revert_page(_uid, 0, 1)
test_eq((_rv['ok'], _rv['version'], _uv.pages(_uid)[0].text), (True, 6, 'the dominat sequence transduktion models'))
test_eq(_uv.revert_page(_uid, 0, 99)['kind'], 'not_found')
test_fail(lambda: _uv.revert_page(_uid, 0, 1, by='agent'), contains='person')
assert 'was' in _ov._proof_t().columns_dict

In [ ]:
#| hide
# an accepted change is made in the page's blocks too, so what is built from them after the proofread has it; an undo takes both back
_kv = Vault(Path(tempfile.mkdtemp())/'k.db', offline=True)
_kid = _kv.read_doc(_src, parse='llm', chat=Boxes('Attention', 'the dominat models'), pages=[0])['doc_id']
_kv.proofread(_kid, ['a', 'b'], pages=[0], chats=dict(a=Boxes('Attention', 'the dominant models'), b=Boxes('Attention', 'the dominant models')))
test_eq(_kv.proofread(_kid, ['a', 'a'], pages=[0], chats=dict(a=Boxes('Attention', 'the dominant models')))['proposed'], 0)   # one reader twice agrees with nobody
_kv.proofread(_kid, ['a', 'b'], pages=[0], chats=dict(a=Boxes('Attention', 'the dominant models'), b=Boxes('Attention', 'the dominant models')))
_kv.accept(_kid)
_p0 = _kv.pages(_kid)[0]
test_eq((_p0.text, [b['text'] for b in _p0.blocks], _p0.blocks[0]['box']), ('## Attention\n\nthe dominant models', ['Attention', 'the dominant models'], [.1, .1, .9, .2]))
_kv.undo_decision(_kid, _kv.changes(_kid, 'accepted').attrgot('id'))
test_eq([b['text'] for b in _kv.pages(_kid)[0].blocks], ['Attention', 'the dominat models'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()